# Trading App V2

This notebook is the live-trading shell for `optimal_trader` going forward.

- `quant-warehouse` owns historical refresh, point-in-time data reads, feature engineering, target engineering, and ThetaData option data.
- `quant-orchestrator` owns feature-family classifier training, option-ranker training, backtests, and strategy artifacts.
- `optimal_trader` owns live broker reconciliation, order planning, Streamlit leaderboard generation, and guarded order submission.

Default behavior builds plans and artifacts only. Orders are submitted exclusively from the generated Streamlit frontend after clicking the Submit Orders button.

In [1]:
from __future__ import annotations

from dataclasses import asdict
from pathlib import Path
import json
import os
import sys

import pandas as pd
from dotenv import load_dotenv
from IPython.display import display

pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 260)
load_dotenv(Path.cwd().resolve() / ".env", override=False)

REPO_ROOT = Path.cwd().resolve()
while not (REPO_ROOT / "app").is_dir() and REPO_ROOT != REPO_ROOT.parent:
    REPO_ROOT = REPO_ROOT.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))


def prefer_local_repo(package_name: str, repo_dir: Path) -> None:
    repo_dir = repo_dir.resolve()
    if not repo_dir.exists():
        return
    repo_text = str(repo_dir)
    sys.path[:] = [
        entry
        for entry in sys.path
        if str(Path(entry or ".").expanduser().resolve()) != repo_text
    ]
    sys.path.insert(0, repo_text)
    loaded = sys.modules.get(package_name)
    loaded_file = Path(str(getattr(loaded, "__file__", ""))).resolve() if loaded is not None else None
    if loaded_file is not None and repo_dir not in loaded_file.parents:
        for module_name in [name for name in sys.modules if name == package_name or name.startswith(f"{package_name}.")]:
            del sys.modules[module_name]


WORKSPACE_ROOT = REPO_ROOT.parent
prefer_local_repo("quant_orchestrator", WORKSPACE_ROOT / "quant-orchestrator")
prefer_local_repo("quant_warehouse", WORKSPACE_ROOT / "quant-warehouse")
prefer_local_repo("fmpsdk", WORKSPACE_ROOT / "fmpsdk")

from app.quant_warehouse_storage import ensure_quant_warehouse_storage
from app.trading_app_v2_runtime import (
    DEFAULT_STRATEGY_SOURCES,
    alpaca_client_from_env,
    build_alpaca_equity_orders,
    build_alpaca_option_orders,
    build_latest_equity_leaderboard,
    build_option_ml_ranking_table,
    backfill_thetadata_eod_for_score_date,
    backfill_thetadata_for_oracle_trade_windows,
    build_score_date_option_ml_ranking_table,
    build_symbol_score_table,
    build_robinhood_option_orders,
    build_llm_review_orders,
    build_llm_ranked_option_orders,
    build_ranked_alpaca_option_orders,
    option_contract_quantity,
    default_paths,
    load_equity_artifacts,
    resolve_option_training_panel,
    save_live_artifacts,
    select_optionable_leaderboard,
    write_streamlit_leaderboard_app,
)

paths = default_paths(REPO_ROOT)
paths.artifact_root.mkdir(parents=True, exist_ok=True)
paths.live_artifact_dir.mkdir(parents=True, exist_ok=True)
paths


TradingAppV2Paths(repo_root=PosixPath('/home/jlee153232/PycharmProjects/optimal_trader'), artifact_root=PosixPath('/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2'), equity_artifact_dir=PosixPath('/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/equity_moe'), option_artifact_dir=PosixPath('/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/option_family_ranker'), live_artifact_dir=PosixPath('/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/live'))

## Environment

This notebook assumes `quant-warehouse`, `quant-orchestrator`, and `tradingagents` are already installed in the `optimal_trader` environment. Dependency installation belongs in environment setup, not in the live trading notebook.


In [2]:
storage = ensure_quant_warehouse_storage()
display(storage.as_dict())


def optional_date(value):
    if value is None:
        return None
    text = str(value).strip()
    if not text or text.lower() in {"none", "null", "nat"}:
        return None
    return text


{'home': '/home/jlee153232/.quant-warehouse',
 'arctic_uri': 'lmdb:///home/jlee153232/.quant-warehouse/arctic',
 'catalog_path': '/home/jlee153232/.quant-warehouse/catalog.sqlite',
 'provider_arctic_uris': {'fmp': 'lmdb:///home/jlee153232/.quant-warehouse/arctic/providers/fmp',
  'thetadata': 'lmdb:///home/jlee153232/.quant-warehouse/arctic/providers/thetadata'}}

## Run Configuration

Shared knobs for universe screening, FMP refresh, feature engineering, target engineering, and downstream training. Feature-family and oracle-label plans are shown after warehouse data is refreshed.


In [3]:
RUN_EQUITY_MOE_TRAINING = os.getenv("TRADING_APP_V2_RUN_EQUITY_TRAINING", "1") == "1"
RUN_OPTION_RANKER_TRAINING = os.getenv("TRADING_APP_V2_RUN_OPTION_TRAINING", "1") == "1"
REBUILD_OPTION_LABEL_PANEL = os.getenv("TRADING_APP_V2_REBUILD_OPTION_LABELS", "1") == "1"
DATA_START = "1900-01-01"
DATA_END = ""  # Empty means latest available downstream data.
MIN_MARKET_CAP = int(os.getenv("TRADING_APP_V2_MIN_MARKET_CAP", "10000000000"))
MIN_FAMILY_TRAIN_ROWS = int(
    os.getenv(
        "TRADING_APP_V2_MIN_FAMILY_TRAIN_ROWS",
        "100" if MIN_MARKET_CAP >= 1_000_000_000_000 else "250",
    )
)
TOP_K = 20  # Use 10 or 20 for live candidate breadth.
MIN_LONG_SCORE = 0.50
REQUIRE_ALL_REQUESTED_FEATURE_FAMILIES = True
REFRESH_MISSING_FMP_DATA = os.getenv("TRADING_APP_V2_REFRESH_FMP", "1") == "1"
REFRESH_MISSING_FMP_INCLUDE_MACRO = True
REFRESH_MISSING_FMP_INCLUDE_PRICES = True
REFRESH_MISSING_FMP_STALENESS_DAYS = 60
REFRESH_MISSING_FMP_SKIP_RECENT_HOURS = 24.0
REFRESH_MISSING_FMP_MAX_WORKERS = 8
REFRESH_THETADATA_SCORE_DATE_EOD = os.getenv("TRADING_APP_V2_REFRESH_OPTIONS", "1") == "1"
THETADATA_OPTION_SCORE_DATE = ""  # Empty means latest equity score_date.
THETADATA_REFRESH_MAX_WORKERS = int(os.getenv("TRADING_APP_V2_OPTION_REFRESH_WORKERS", "1"))
THETADATA_REFRESH_OVERWRITE = False
REFRESH_THETADATA_ORACLE_TRADE_WINDOWS = os.getenv("TRADING_APP_V2_REFRESH_OPTION_TRAINING", "1") == "1"
REFRESH_THETADATA_ORACLE_TRADE_RANGES = os.getenv("TRADING_APP_V2_REFRESH_OPTION_RANGES", "1") == "1"
THETADATA_ORACLE_BACKFILL_MAX_TRADES = int(os.getenv("TRADING_APP_V2_OPTION_BACKFILL_MAX_TRADES", "25"))
THETADATA_ORACLE_BACKFILL_WINDOW_DAYS = 7
THETADATA_ORACLE_FALLBACK_WINDOW_DAYS = 1
THETADATA_ORACLE_BACKFILL_REQUEST_SLEEP = 0.0
THETADATA_ORACLE_BACKFILL_OVERWRITE = False
RUN_BACKTESTS_IN_ORCHESTRATOR = False

ALPACA_EQUITY_ACCOUNT_PREFIX = "EQUITY"
ALPACA_OPTION_ACCOUNT_PREFIX = "OPTION"
ALPACA_LLM_ACCOUNT_PREFIX = "LLM"

OPTION_STRATEGY_ALLOCATION = 100_000.0
OPTION_BUCKET = "otm_option"
OPTION_TENOR_DAYS = 90

# Real Robinhood follows the option strategy with deeply discounted GTC limits.
ROBINHOOD_OPTION_GATE_DISCOUNT_PCT = 90.0

# Equity and LLM real-account gates remain blocked until paper PnL justifies changing them.
ROBINHOOD_EQUITY_GATE_DISCOUNT_PCT = 100.0
ROBINHOOD_LLM_GATE_DISCOUNT_PCT = 100.0

OPTION_PANEL_FOR_RANKER = resolve_option_training_panel(
    paths.option_artifact_dir,
    min_market_cap=MIN_MARKET_CAP,
)

display(
    {
        "artifact_root": str(paths.artifact_root),
        "equity_artifact_dir": str(paths.equity_artifact_dir),
        "option_artifact_dir": str(paths.option_artifact_dir),
        "live_artifact_dir": str(paths.live_artifact_dir),
        "data_start": DATA_START,
        "min_market_cap": MIN_MARKET_CAP,
        "refresh_missing_fmp_data": REFRESH_MISSING_FMP_DATA,
    }
)


{'artifact_root': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2',
 'equity_artifact_dir': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/equity_moe',
 'option_artifact_dir': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/option_family_ranker',
 'live_artifact_dir': '/home/jlee153232/PycharmProjects/optimal_trader/artifacts/trading_app_v2/live',
 'data_start': '1900-01-01',
 'min_market_cap': 10000000000,
 'refresh_missing_fmp_data': True}

## FMP Historical Refresh

Refresh raw FMP warehouse data before feature engineering and target engineering. This cell screens the universe first, displays the exact symbols that will be used downstream, then refreshes only those symbols in the shared Quant Warehouse store. The training cell below disables internal refresh so model fitting starts only after this visible refresh step completes.


In [4]:
from quant_warehouse.migrate.backfill_missing_fmp import backfill_missing_fmp_historical
from quant_warehouse.research_tools.feature_family_eval import FamilyEvaluationConfig, screen_fmp_equity_universe
from quant_warehouse.warehouse.api import Warehouse

refresh_warehouse = Warehouse()
refresh_feature_config = FamilyEvaluationConfig(
    provider="fmp",
    market_cap_min=MIN_MARKET_CAP,
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
)

screened_equity_symbols, raw_fmp_universe, universe_eligibility, universe_source = screen_fmp_equity_universe(
    refresh_feature_config,
    warehouse=refresh_warehouse,
)
screened_symbols_df = pd.DataFrame({"symbol": list(screened_equity_symbols)})
print(
    f"FMP universe source={universe_source}; "
    f"eligible_symbols={len(screened_equity_symbols):,}; "
    f"min_market_cap={MIN_MARKET_CAP:,}"
)
display(screened_symbols_df)
print("Universe eligibility sample")
display(universe_eligibility.head(100))


def notebook_refresh_log(message: str) -> None:
    print(f"[fmp-refresh] {message}", flush=True)


fmp_refresh_summary = {"status": "skipped_by_config", "equity_symbols": list(screened_equity_symbols), "etf_symbols": []}
if REFRESH_MISSING_FMP_DATA:
    fmp_refresh_summary = backfill_missing_fmp_historical(
        warehouse=refresh_warehouse,
        equity_provider="fmp",
        etf_provider="fmp",
        equity_symbols=screened_equity_symbols,
        etf_symbols=(),
        include_macro=REFRESH_MISSING_FMP_INCLUDE_MACRO,
        include_prices=REFRESH_MISSING_FMP_INCLUDE_PRICES,
        staleness_days=REFRESH_MISSING_FMP_STALENESS_DAYS,
        skip_recent_hours=REFRESH_MISSING_FMP_SKIP_RECENT_HOURS,
        max_workers=REFRESH_MISSING_FMP_MAX_WORKERS,
        progress_logger=notebook_refresh_log,
    )
else:
    print("FMP refresh skipped because REFRESH_MISSING_FMP_DATA=False")

def macro_refresh_status(macro_payload) -> str | None:
    if isinstance(macro_payload, dict):
        return macro_payload.get("status")
    if isinstance(macro_payload, list):
        statuses = [
            str(item.get("status"))
            for item in macro_payload
            if isinstance(item, dict) and item.get("status") is not None
        ]
        if not statuses:
            return None
        counts = pd.Series(statuses).value_counts()
        return ", ".join(f"{name}={count}" for name, count in counts.items())
    return None


fmp_refresh_summary_path = paths.equity_artifact_dir / "fmp_refresh_summary.json"
fmp_refresh_summary_path.parent.mkdir(parents=True, exist_ok=True)
fmp_refresh_summary_path.write_text(json.dumps(fmp_refresh_summary, indent=2, default=str), encoding="utf-8")
print(f"Wrote FMP refresh summary to {fmp_refresh_summary_path}")
display(pd.DataFrame([{
    "equity_symbols": len(fmp_refresh_summary.get("equity_symbols", [])),
    "etf_symbols": len(fmp_refresh_summary.get("etf_symbols", [])),
    "equity_price_total": fmp_refresh_summary.get("equity_prices", {}).get("total"),
    "equity_price_updated": fmp_refresh_summary.get("equity_prices", {}).get("updated"),
    "equity_fundamental_total": fmp_refresh_summary.get("equity", {}).get("total"),
    "equity_fundamental_updated": fmp_refresh_summary.get("equity", {}).get("updated"),
    "macro_status": macro_refresh_status(fmp_refresh_summary.get("macro")),
}]))


FMP universe source=openbb:fmp+catalog:fmp:fallback_after_OpenBBError+catalog:fmp:fallback_after_EmptyDataError; eligible_symbols=801; min_market_cap=10,000,000,000


,symbol
0,A
1,AA
2,AAL
3,AAPL
4,ABBV
...,...
796,ZBRA
797,ZION
798,ZM
799,ZS


Universe eligibility sample


,symbol,eligible,reason,screen_market_cap
0,NVDA,True,ok,4.929700e+12
1,AAPL,True,ok,4.660445e+12
2,GOOG,True,ok,4.266993e+12
3,GOOGL,True,ok,4.263563e+12
4,MSFT,True,ok,2.904442e+12
...,...,...,...,...
95,RVMD,True,ok,3.910703e+10
96,NTRA,True,ok,3.908194e+10
97,ALNY,True,ok,3.845963e+10
98,WDAY,True,ok,3.794755e+10


[fmp-refresh] Backfill: scoped equity symbols (801): A, AA, AAL, AAPL, ABBV, ABC, ABNB, ABT, ACN, ADBE, ADI, ADM, ADP, ADSK, AEE, AEIS, AEP, AES, AFG, AFL, AFRM, AGNC, AGR, AGX, AHR, AIG, AIT, AIZ, AIZN, AJG, AKAM, ALAB, ALB, ALGN, ALL, ALLE, ALLY, ALNY, AM, AMAT, AMD, AME, AMGN, AMH, AMKR, AMP, AMT, AMZN, ANET, ANTM, AON, APA, APC, APD, APG, APH, APO, APOS, APP, APTV, AQNB, AR, ARCC, ARES, ARMK, ARW, ARWR, ASTS, ATHS, ATI, ATO, AUR, AVB, AVGO, AVY, AWK, AXON, AXP, AXSM, AYI, AZO, BA, BAC, BALL, BAX, BBIO, BBY, BDX, BE, BEN, BEPH, BEPI, BF-B, BG, BIIB, BIPI, BJ, BK, BKDT, BKI, ... (+701 more)
[fmp-refresh] Backfill: refreshing equity prices for 767 stale symbols (801 scoped) via FMP
[fmp-refresh] Warehouse price refresh progress: 1/767 symbols processed
[fmp-refresh] Warehouse price refresh progress: 25/767 symbols processed
[fmp-refresh] Warehouse price refresh progress: 50/767 symbols processed
[fmp-refresh] Warehouse price refresh progress: 75/767 symbols processed
[fmp-refresh] War

,equity_symbols,etf_symbols,equity_price_total,equity_price_updated,equity_fundamental_total,equity_fundamental_updated,macro_status
0,801,0,767,759,7560,59,skipped_complete


## Target Engineering

Build collapsed oracle trade labels from refreshed warehouse prices. This notebook uses `oracle_only` labels: side-specific `YE` oracle entries for `k=1..12`, collapsed into `oracle_long` and `oracle_short` classes.


In [5]:
from quant_warehouse.research_tools import BinaryTargetConfig
from quant_warehouse.platforms.data_providers.fmp.target_engineering import LabelBuildSpec, build_trade_results
from quant_warehouse.warehouse.api import Warehouse
from quant_orchestrator.research_tools.ml_trading_experiment import _build_oracle_trade_label_rows_sparse

if "screened_equity_symbols" not in globals():
    raise RuntimeError("Run the FMP Historical Refresh cell before target engineering.")

target_warehouse = Warehouse()

target_config = BinaryTargetConfig(
    provider="fmp",
    start_date=DATA_START,
    end_date=optional_date(DATA_END),
    oracle_trade_k_by_frequency={"YE": tuple(range(1, 4))},
)

oracle_label_rows, oracle_label_diagnostics, oracle_seconds, oracle_trade_windows_unique = _build_oracle_trade_label_rows_sparse(
    screened_equity_symbols,
    target_config,
    warehouse=target_warehouse,
)

oracle_price_frames = {}
for raw_symbol in screened_equity_symbols:
    symbol = str(raw_symbol).strip().upper()
    if not symbol:
        continue
    prices = target_warehouse.read_prices(
        symbol,
        provider=target_config.provider,
        start=target_config.start_date,
        end=target_config.end_date,
    )
    if prices is not None and not prices.empty:
        oracle_price_frames[symbol] = prices

oracle_trade_spec = LabelBuildSpec(
    k_params={frequency: list(ks) for frequency, ks in target_config.oracle_trade_k_by_frequency.items()},
    min_profit_pct=target_config.oracle_trade_min_profit_pct,
    buy_execution=target_config.oracle_trade_long_entry_price_col,
    sell_execution=target_config.oracle_trade_long_exit_price_col,
    short_execution=target_config.oracle_trade_short_entry_price_col,
    cover_execution=target_config.oracle_trade_short_exit_price_col,
    start_date=target_config.start_date,
    end_date=target_config.end_date,
)
oracle_trade_result = build_trade_results(
    screened_equity_symbols,
    spec=oracle_trade_spec,
    price_frames=oracle_price_frames,
)
oracle_trade_windows = pd.DataFrame(oracle_trade_result.completed_trades)
if not oracle_trade_windows.empty:
    oracle_trade_windows["symbol"] = oracle_trade_windows["symbol"].astype(str).str.upper()
    oracle_trade_windows["entry_date"] = pd.to_datetime(oracle_trade_windows["entry_date"], errors="coerce").dt.normalize()
    oracle_trade_windows["exit_date"] = pd.to_datetime(oracle_trade_windows["exit_date"], errors="coerce").dt.normalize()
    oracle_trade_windows = oracle_trade_windows.dropna(subset=["symbol", "entry_date", "exit_date"])
    if "trade_id" not in oracle_trade_windows.columns:
        oracle_trade_windows["trade_id"] = (
            oracle_trade_windows["symbol"].astype(str)
            + "|"
            + oracle_trade_windows["side"].astype(str)
            + "|"
            + oracle_trade_windows["entry_date"].dt.strftime("%Y%m%d")
            + "|"
            + oracle_trade_windows["exit_date"].dt.strftime("%Y%m%d")
            + "|k"
            + oracle_trade_windows["k"].astype(str)
        )
    oracle_trade_windows = oracle_trade_windows.sort_values(
        ["entry_date", "symbol", "trade_id"],
        ascending=[False, True, True],
        kind="stable",
    ).reset_index(drop=True)

print(
    {
        "oracle_label_rows": len(oracle_label_rows),
        "oracle_classes": sorted(oracle_label_rows["collapsed_label"].dropna().unique())
        if not oracle_label_rows.empty
        else [],
        "oracle_seconds": round(oracle_seconds, 3),
        "oracle_trade_windows": len(oracle_trade_windows),
    }
)
display(oracle_label_diagnostics)
display(oracle_label_rows.head(20))

{'oracle_label_rows': 153085, 'oracle_classes': ['oracle_long', 'oracle_short'], 'oracle_seconds': 36.722, 'oracle_trade_windows': 306476}


,source,candidate_rows,used_rows,dropped_rows,unique_trade_windows,ambiguous_entry_dates,note
0,oracle_trade,306476,153085,115019,191457,0,"unique windows on (symbol, side, entry_date, e..."


,symbol,date,collapsed_label,label_source
0,A,1999-11-26,oracle_long,oracle_trade
1,A,1999-12-15,oracle_long,oracle_trade
2,A,2000-01-06,oracle_long,oracle_trade
3,A,2000-05-24,oracle_long,oracle_trade
4,A,2000-08-08,oracle_long,oracle_trade
5,A,2001-01-02,oracle_long,oracle_trade
6,A,2001-04-06,oracle_long,oracle_trade
7,A,2001-09-27,oracle_long,oracle_trade
8,A,2002-01-02,oracle_long,oracle_trade
9,A,2002-02-08,oracle_long,oracle_trade


## ThetaData Oracle Backfill

Backfill policy is configurable: `oracle_entry_exit` refreshes only trade endpoints, while `all` refreshes every missing business date in each oracle window. This notebook uses `all`, newest dates first, so reruns progressively work backward. Each symbol/date is checked against the Arctic cache before any ThetaData request is made.


In [ ]:
from quant_warehouse.migrate.backfill_thetadata_options import (
    normalize_oracle_trade_windows,
    backfill_thetadata_options_for_oracle_trade_ranges,
    fmp_trading_days_for_year,
)

if "oracle_trade_windows" not in globals() or oracle_trade_windows.empty:
    raise RuntimeError("Target engineering must produce oracle_trade_windows before ThetaData refresh.")

oracle_backfill_symbols = tuple(screened_equity_symbols) if "screened_equity_symbols" in globals() else ()
theta_oracle_trade_windows = normalize_oracle_trade_windows(
    oracle_trade_windows,
    symbols=oracle_backfill_symbols,
)
if theta_oracle_trade_windows.empty:
    raise RuntimeError("No normalized oracle trade windows are available for ThetaData refresh.")

THETADATA_ORACLE_BACKFILL_MODE = "all"  # also supported: "oracle_entry_exit"
if THETADATA_ORACLE_BACKFILL_MODE not in {"oracle_entry_exit", "all"}:
    raise ValueError("THETADATA_ORACLE_BACKFILL_MODE must be 'oracle_entry_exit' or 'all'")

print(f"Refreshing ThetaData in yearly phases, newest year first; endpoint mode={THETADATA_ORACLE_BACKFILL_MODE!r}.")
theta_years = sorted(
    {int(year) for year in theta_oracle_trade_windows["entry_date"].dt.year.tolist() + theta_oracle_trade_windows["exit_date"].dt.year.tolist()},
    reverse=True,
)
thetadata_oracle_refresh_summary = {"status": "ok", "years": [], "dates_downloaded": 0, "dates_failed": 0}
thetadata_probed_symbols = set()
for theta_year in theta_years:
    year_start = pd.Timestamp(f"{theta_year}-01-01")
    year_end = pd.Timestamp(f"{theta_year}-12-31")
    theta_fmp_trading_days = fmp_trading_days_for_year(theta_year, warehouse=target_warehouse)
    print(f"FMP trading-day calendar: {len(theta_fmp_trading_days):,} dates for {theta_year}")
    endpoint_windows = theta_oracle_trade_windows.loc[
        theta_oracle_trade_windows["entry_date"].dt.year.eq(theta_year)
        | theta_oracle_trade_windows["exit_date"].dt.year.eq(theta_year)
    ].copy()
    year_windows = theta_oracle_trade_windows.loc[
        theta_oracle_trade_windows["entry_date"].le(year_end)
        & theta_oracle_trade_windows["exit_date"].ge(year_start)
    ].copy()
    year_windows["entry_date"] = year_windows["entry_date"].clip(lower=year_start)
    year_windows["exit_date"] = year_windows["exit_date"].clip(upper=year_end)
    print(f"\n===== ThetaData year {theta_year}: endpoint phase ({len(endpoint_windows):,} trades) =====")
    endpoint_summary = backfill_thetadata_options_for_oracle_trade_ranges(
        endpoint_windows,
        mode="oracle_entry_exit",
        symbols=oracle_backfill_symbols,
        max_trades=None,
        trading_days=theta_fmp_trading_days,
        probed_symbols=thetadata_probed_symbols,
        skip_existing=True,
        overwrite=False,
        request_sleep=0.0,
        progress_logger=print,
    )
    all_summary = {"status": "not_selected"}
    if THETADATA_ORACLE_BACKFILL_MODE == "all":
        print(f"===== ThetaData year {theta_year}: all-date phase ({len(year_windows):,} windows) =====")
        all_summary = backfill_thetadata_options_for_oracle_trade_ranges(
            year_windows,
            mode="all",
            symbols=oracle_backfill_symbols,
            max_trades=None,
            trading_days=theta_fmp_trading_days,
            probed_symbols=thetadata_probed_symbols,
            skip_existing=True,
            overwrite=False,
            request_sleep=0.0,
            progress_logger=print,
        )
    thetadata_oracle_refresh_summary["years"].append({"year": theta_year, "endpoint": endpoint_summary, "all": all_summary})
    thetadata_oracle_refresh_summary["dates_downloaded"] += int(all_summary.get("dates_downloaded", 0))
    thetadata_oracle_refresh_summary["dates_failed"] += int(all_summary.get("dates_failed", 0))
display({k: v for k, v in thetadata_oracle_refresh_summary.items() if k != "years"})
display(pd.DataFrame([{"year": x["year"], "endpoint_dates_downloaded": x["endpoint"].get("trade_windows_completed", 0), "all_dates_requested": x["all"].get("dates_requested", 0), "all_dates_downloaded": x["all"].get("dates_downloaded", 0), "all_dates_failed": x["all"].get("dates_failed", 0)} for x in thetadata_oracle_refresh_summary["years"]]))
